In [1]:
import math
import numpy as np
import pandas as pd
import plotly.express as px
import pickle

In [2]:
# Load the training and test datasets
train_data = pd.read_csv('/kaggle/input/datasets/vidyaareddi/linearregtest/train.csv')
test_data = pd.read_csv('/kaggle/input/datasets/vidyaareddi/linearregtest/test.csv')

# Remove rows with missing values
train_data = train_data.dropna()
test_data = test_data.dropna()

In [3]:
train_data.head(4)

,x,y
0,24.0,21.549452
1,50.0,47.464463
2,15.0,17.218656
3,38.0,36.586398


In [4]:
px.scatter(x=train_data['x'], y=train_data['y'],template='seaborn')

In [5]:
xtrain=train_data['x'].values
ytrain=train_data['y'].values
xtest = test_data['x'].values
ytest = test_data['y'].values

Standardization is a preprocessing technique used in machine learning to rescale and transform the features (variables) of a dataset to have a mean of 0 and a standard deviation of 1. It is also known as "z-score normalization" or "z-score scaling." 

Feature Standardization (Z-score Normalization)

Standardization transforms features so they have:
- Mean = 0  
- Standard deviation = 1  

---


1. Compute the mean and standard deviation for each feature:

$$
\mu = \text{mean of feature}, \quad \sigma = \text{standard deviation}
$$

2. Transform each value:

- Subtract the mean  
- Divide by standard deviation  

---

Formula

$$
x_{\text{standardized}} = \frac{x - \mu}{\sigma}
$$

---

Where:

- \( x \): original feature value  
- \( \mu \): mean of the feature  
- \( \sigma \): standard deviation of the feature  

---

Intuition

- Centers data around 0  
- Scales data to unit variance  
- Helps models like Linear Regression converge faster  
``

In [6]:
def standardiseData(xtrain,xtest):
    mean=np.mean(xtrain,axis=0)
    std=np.std(xtrain,axis=0)
    xtrain=(xtrain-mean)/std
    xtest=(xtest-mean)/std
    return xtrain,xtest

xtrain,xtest=standardiseData(xtrain,xtest)

In [7]:
xtrain = np.expand_dims(xtrain, axis=-1)
xtest = np.expand_dims(xtest, axis=-1)

In [8]:
import numpy as np


class LinearRegression:
    def __init__(self,learning_rate,convergence_tol=1e-6):
        self.learning_rate = learning_rate
        self.convergence_tol = convergence_tol
        self.W = None
        self.b = None

    def initialize_parameters(self,n_features):
        self.W = np.random.randn(n_features)*0.01
        self.b = 0

    def forward(self,x):
        return np.dot(x,self.W)+self.b

    def compute_cost(self,predictions):
        m=len(predictions)
        cost = np.sum(np.square(predictions - self.y)) / (2 * m)
        return cost
    def backward(self,predictions):
        m=len(predictions)
        self.dW = np.dot(predictions - self.y, self.x) / m
        self.db = np.sum(predictions - self.y) / m

    def fit(self, x, y, iterations, plot_cost=True):
        assert isinstance(x, np.ndarray), "X must be a NumPy array"
        assert isinstance(y, np.ndarray), "y must be a NumPy array"
        assert x.shape[0] == y.shape[0], "X and y must have the same number of samples"
        assert iterations > 0, "Iterations must be greater than 0"

        self.x = x
        self.y = y
        self.initialize_parameters(x.shape[1])
        costs = []

        for i in range (iterations):
            predictions = self.forward(x)
            cost=self.compute_cost(predictions)
            self.backward(predictions)

            self.W -= self.learning_rate * self.dW
            self.b -= self.learning_rate * self.db
            costs.append(cost)

            if i % 100 == 0:
                print(f'Iteration: {i}, Cost: {cost}')

            if i > 0 and abs(costs[-1] - costs[-2]) < self.convergence_tol:
                print(f'Converged after {i} iterations.')
                break

    def predict(self,x):
        return self.forward(x)

    def save_model(self, filename=None):
        model_data = {
            'learning_rate': self.learning_rate,
            'convergence_tol': self.convergence_tol,
            'W': self.W,
            'b': self.b
        }

        with open(filename, 'wb') as file:
            pickle.dump(model_data, file)

    @classmethod
    def load_model(cls, filename):
        with open(filename, 'rb') as file:
            model_data = pickle.load(file)

        # Create a new instance of the class and initialize it with the loaded parameters
        loaded_model = cls(model_data['learning_rate'], model_data['convergence_tol'])
        loaded_model.W = model_data['W']
        loaded_model.b = model_data['b']

        return loaded_model

In [9]:
lr = LinearRegression(0.01)
lr.fit(xtrain, ytrain, 10000)

Iteration: 0, Cost: 1670.268858532737
Iteration: 100, Cost: 227.1888954817913
Iteration: 200, Cost: 33.84551123818606
Iteration: 300, Cost: 7.941427481289751
Iteration: 400, Cost: 4.470806762047374
Iteration: 500, Cost: 4.005814126527968
Iteration: 600, Cost: 3.9435145644097287
Iteration: 700, Cost: 3.9351676893333347
Iteration: 800, Cost: 3.9340493777245205
Converged after 863 iterations.


In [10]:
lr.save_model('model.pkl')

In [11]:
model = LinearRegression.load_model("model.pkl")

In [12]:
class RegressionMetrics:
    @staticmethod
    def mean_squared_error(y_true, y_pred):
        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mse = np.mean((y_true - y_pred) ** 2)
        return mse

    @staticmethod
    def root_mean_squared_error(y_true, y_pred):
        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mse = RegressionMetrics.mean_squared_error(y_true, y_pred)
        rmse = np.sqrt(mse)
        return rmse

    @staticmethod
    def r_squared(y_true, y_pred):
        assert len(y_true) == len(y_pred), "Input arrays must have the same length."
        mean_y = np.mean(y_true)
        ss_total = np.sum((y_true - mean_y) ** 2)
        ss_residual = np.sum((y_true - y_pred) ** 2)
        r2 = 1 - (ss_residual / ss_total)
        return r2

In [13]:
ypred = model.predict(xtest)
mse_value = RegressionMetrics.mean_squared_error(ytest, ypred)
rmse_value = RegressionMetrics.root_mean_squared_error(ytest, ypred)
r_squared_value = RegressionMetrics.r_squared(ytest, ypred)

print(f"Mean Squared Error (MSE): {mse_value}")
print(f"Root Mean Squared Error (RMSE): {rmse_value}")
print(f"R-squared (Coefficient of Determination): {r_squared_value}")

Mean Squared Error (MSE): 9.442670818162243
Root Mean Squared Error (RMSE): 3.0728929070441495
R-squared (Coefficient of Determination): 0.9887898710804992
